In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
import numpy as np
import json
import folium

This notebook contains all the parts of the code needed to create the route based on the preprocessed and prepared data from the previous notebook. For this, I need to load in all the necessary data and collect certain input from the user about preferences and length of stay. Then I can filter my data accordingly and create an algorithm that creates an itinerary and route, using some input parameters. Finally, I create a visualization of this.

## Step 1 - Loading all necessary data

In [2]:
# road network

In [3]:
G = ox.load_graphml('iceland_drive_accessible.graphml')

In [4]:
G_undirected = ox.convert.to_undirected(G)

In [5]:
# distance matrix

In [6]:
dist_df = pd.read_csv('distance_matrix.csv', index_col=0)

In [7]:
# convert to int
dist_df.index = dist_df.index.map(lambda x: int(float(x)))
dist_df.columns = dist_df.columns.map(lambda x: int(float(x)))

In [8]:
# deduplicate
dist_df = dist_df[~dist_df.index.duplicated(keep='first')]
dist_df = dist_df.loc[:, ~dist_df.columns.duplicated(keep='first')]

In [9]:
# keflavik node not included with other attractions

In [10]:
# Load distances from Keflavik for all attractions
with open('config.json', 'r') as f:
    config = json.load(f)

keflavik_node = config['keflavik_node']

In [11]:
keflavik_node

5174792758

In [12]:
# attractions df

In [13]:
# read in and split categories

alternative_attractions_df = gpd.read_file('alternative_attractions_processed.geojson')
alternative_attractions_df['category'] = alternative_attractions_df['category'].str.split('•').apply(lambda x: [i.strip() for i in x] if isinstance(x, list) else x)
alternative_attractions_df['group'] = alternative_attractions_df['group'].str.split('•').apply(lambda x: [i.strip() for i in x] if isinstance(x, list) else x)

In [14]:
alternative_attractions_df.head(1)

,name,link,rating,review_count,category,address,latitude,longitude,X_metres,Y_metres,...,nights,population,area_km2,stays_per_pop,stays_per_area,x,y,node_id,hiddengem_score,geometry
0,Solheimajokull Glacier,https://www.tripadvisor.com/Attraction_Review-...,4.7,864,[Geologic Formations],NaN,63.534584,-19.356466,482263.853808,336679.637255,...,444788.0,881.0,749.097377,504.867196,593.765261,-19.356466,63.534584,3762059788,-1.110223e-16,POINT (-19.35647 63.53458)


## Step 2 - Collect inputs from user & set key parameters

The user can indicate preferences of types of attractions and define how many days they will be travelling for. 
I also set some parameters that the user has no influence over.

In [15]:
# set params - no user choice

DAILY_BUDGET_KM = 300
# to imitate a realistic way of traveling, I am limiting the maximum amount of stops visited per day to 5 to allow the user to spend some 
# time at each
MAX_STOPS_PER_DAY = 5

# Routing parameters
# anchors will be used in routing as bases per day/the main attraction for the day, which sets the geographic direction of the day
TOP_N_ANCHORS = 10
# I predefine how far away secondary attractions for the day can be from the anchor
SECONDARY_RADIUS_KM = 50
# if there are too little reachable, I extend the raduis
SECONDARY_RADIUS_EXPANDED_KM = 80
# I always want to have at least 2 secondary attractions that can be visited
MIN_SECONDARIES = 2

In [16]:
# for now for developing the prototype, I will hardcode this

N_DAYS = 10
SELECTED_GROUPS = ['Nature/Adventure', 'Sightseeing']
RANDOM_SEED = 42

In [17]:
# for streamlit app

'''

# Parameters 
N_DAYS = st.slider("How many days is your trip?", min_value=3, max_value=10, value=5)

SELECTED_GROUPS = st.multiselect(
    "What kind of experiences are you interested in?",
    options=['Nature/Adventure', 'Sightseeing', 'Culture', 'Services', 'Leisure'],
    default=['Nature/Adventure', 'Sightseeing']
)

# random seed ensures the route is different every time
RANDOM_SEED = random.randint(0, 1000)

st.button("Generate Route")

# force user to select at least one group of attractions
if len(SELECTED_GROUPS) == 0:
    st.warning("Please select at least one category of interest.")
    st.stop()
'''

'\n\n# Parameters \nN_DAYS = st.slider("How many days is your trip?", min_value=3, max_value=10, value=5)\n\nSELECTED_GROUPS = st.multiselect(\n    "What kind of experiences are you interested in?",\n    options=[\'Nature/Adventure\', \'Sightseeing\', \'Culture\', \'Services\', \'Leisure\'],\n    default=[\'Nature/Adventure\', \'Sightseeing\']\n)\n\n# random seed ensures the route is different every time\nRANDOM_SEED = random.randint(0, 1000)\n\nst.button("Generate Route")\n\n# force user to select at least one group of attractions\nif len(SELECTED_GROUPS) == 0:\n    st.warning("Please select at least one category of interest.")\n    st.stop()\n'

## Step 3 - Filter df based on user's input

In [18]:
df_filtered = alternative_attractions_df[
    alternative_attractions_df['group'].apply(
        lambda x: any(g in SELECTED_GROUPS for g in x) if isinstance(x, list) else x in SELECTED_GROUPS)].copy()

In [19]:
print(f"Attractions after category filter: {len(df_filtered)}")

Attractions after category filter: 407


In [20]:
# Warn if too few attractions
if len(df_filtered) < N_DAYS * 3:
    print(f"Warning: only {len(df_filtered)} attractions found for selected categories. Consider broadening your selection.")

In [21]:
# for streamlit app

'''
if len(df_filtered) < N_DAYS * 3:
    st.warning(f"Only {len(df_filtered)} attractions found for your selected categories. Consider broadening your selection.")
    st.stop()
'''

'\nif len(df_filtered) < N_DAYS * 3:\n    st.warning(f"Only {len(df_filtered)} attractions found for your selected categories. Consider broadening your selection.")\n    st.stop()\n'

## Step 4 - Filter attractions based on reachability within trip

Depending on the length of the trip, not all attractions can be reached using the 300km per day driving budget. To avoid issues when routing, I will filter out those that cannot be reached anymore.

In [22]:
# the route always has to be able to reach the airport again on the next day, so enough 'budget' needs to be left for that
# the distance matrix is in meter, so I need to convert km to m
# considering that each route is for a round trip, the user can never go further than half of the total budget away

total_budget_m = N_DAYS * DAILY_BUDGET_KM * 1000
max_reach_m = (total_budget_m / 2)

# get distances from Keflavik for all attractions
kef_row = dist_df.loc[keflavik_node]
kef_row = kef_row[~kef_row.index.duplicated(keep='first')]
kef_dist_dict = {int(float(k)): v for k, v in kef_row.to_dict().items()}

# map distance from Keflavik Airport to each attraction in the dataframe
df_filtered['dist_to_kef'] = df_filtered['node_id'].map(kef_dist_dict)

# filter out those that are too far away to reach within the whole trip
df_filtered = df_filtered[df_filtered['dist_to_kef'] <= max_reach_m].copy()

In [23]:
print(f"Max reachable distance: {max_reach_m/1000:.0f}km")

Max reachable distance: 1500km


In [24]:
len(df_filtered)

407

In [25]:
print(f"Max distance in filtered set: {df_filtered['dist_to_kef'].max()/1000:.0f}km")
print(f"Min distance in filtered set: {df_filtered['dist_to_kef'].min()/1000:.0f}km")

Max distance in filtered set: 733km
Min distance in filtered set: 2km


## Step 5 - Initalize routing variables

In [26]:
# set up all the variables needed for and in the routing process

# always start at the airport
current_node = keflavik_node
visited_ids = set()
itinerary = []
remaining_days = N_DAYS
# keeping track of all anchors to ensure geographic diversity
anchor_nodes_visited = [keflavik_node]

## Step 6 - Routing Algorithm

With the above as a base, I can now run the routing algorithm itself, which creates a route day by day and thus slowly fills up the itinerary.

The algorithm contains five steps to create the route for each day.

1. I check for all attractions where they can still be reached within the budget of that day and allow to return to the airport within the overall budget.
2. I ensure that there are no 'dead ends' in my routing, which are too far from any other points to move on within the driving budget.
3. I pick the anchors for my routing. They are the main attraction of each day, around which all attractions for the day are centered. I also hold the driving budget against the distance between the anchors. I am forcing anchors to be at least 100km apart to ensure geographic diversity in my route.
4. Then I pick secondary attraction around the days anchor.
5. Finally, I add things to the itinerary and reset them for the next day.

In [27]:
for day in range(N_DAYS):

    # STEP 1
    # check elegibility of attractions for this day based on whether the can be reached within the days budget and
    # allow to return to the airport within the rest of the trips budget

    if remaining_days == 1:
        max_dist_from_kef = DAILY_BUDGET_KM * 1000
    else:
        max_dist_from_kef = (remaining_days - 1) * DAILY_BUDGET_KM * 1000

    # to be an eligible attractions, three criteria must be fulfilled
    eligible = df_filtered[
        # only go to attractions that have not already been visited
        (~df_filtered.index.isin(visited_ids)) &
        # check that the user must be able to return to Keflavik within remaining days
        (df_filtered['dist_to_kef'] <= max_dist_from_kef) &
        # reachable from current position today
        (df_filtered['node_id'].map(lambda n: dist_df.at[current_node, n]) <= DAILY_BUDGET_KM * 1000)].copy()


    # STEP 2
    # to ensure the user does not get stuck in at an attractions that is too far away from other to reach other attractions 
    # we need to exclude those, unless it is the last day for which we already ensure the airport can be reached again above
    # likely this will not do much but works as a safety net to ensure my routing will not get stuck
    if day < N_DAYS - 1:
        def has_onward_options(node):
            # count how many unvisited eligible attractions are reachable from this node
            reachable_from_node = df_filtered[
                (~df_filtered.index.isin(visited_ids)) &
                (df_filtered['node_id'].map(lambda n: dist_df.at[node, n]) <= DAILY_BUDGET_KM * 1000)
            ]
            # returns true if there is at least one other reachable attraction from the current one
            return len(reachable_from_node) > 0
        
        eligible = eligible[eligible['node_id'].map(has_onward_options)].copy()


    # STEP 3
    # my routing uses anchors, meaning that each day is centered around a main attraction, which I call the anchor,
    # and then some other attractions around it
    # this is a fairly easy way to create a realistic route
    # to decide on the anchors, I use the hidden gem score I created above and add some randomness to ensure different attractions
    # will be picked as anchors
    # however, I also do not want the user to stay in the same area each day, so I will force the anchors starting from the second day
    # to at least be 100km away from the current position

    # first I need to calculate distance from current position for all eligible attractions
    distance_from_current = eligible['node_id'].map(lambda n: dist_df.at[current_node, n])
    
    # then I apply a minimum distance on days 2 to N-1 of 100km
    # this value is a bit arbitrary, but distances between regions in Iceland are fairly big
    # on day 1, 100km might be too far, since Reykjavik for example is only about 50km from the airport
    if 1 <= day <= N_DAYS - 2:
        # from days 2 to the second-to-last I enforce a 100km minimum
        eligible_band = eligible[
            (distance_from_current >= 100 * 1000) &
            (distance_from_current <= DAILY_BUDGET_KM * 1000)
        ].copy()
    elif day == N_DAYS - 1:
        # on the last day I enforce a softer 50km minimum so route doesn't stay in same spot but 
        # can still be created with all other limitations put on it
        eligible_band = eligible[
            (distance_from_current >= 50 * 1000) &
            (distance_from_current <= DAILY_BUDGET_KM * 1000)
        ].copy()
    else:
        # day 1 has no minimum
        eligible_band = eligible

    # to enforce diversity not just between days but overall, I will check if my anchor node is too close to any of the previous anchors
    def too_close_to_previous_anchors(node):
        return any(
            dist_df.at[node, prev_anchor] < 50 * 1000
            for prev_anchor in anchor_nodes_visited
        )
    
    eligible_band = eligible_band[
        ~eligible_band['node_id'].map(too_close_to_previous_anchors)
    ].copy()
    
    # should this cause trouble, I fallback as above
    if len(eligible_band) < TOP_N_ANCHORS:
        eligible_band = eligible

    # then I take the top N candidates by hidden gem score
    # I have defined in my set parameters above that I will pick the top 10 
    top_candidates = eligible_band.nlargest(TOP_N_ANCHORS, 'hiddengem_score')

    # ensure that routing does not break if no more attractions are available
    if len(top_candidates) == 0:
        print(f"Day {day+1}: no eligible attractions, ending itinerary")
        break

    # weighted random selection using hidden gem scores as weights
    weights = top_candidates['hiddengem_score'].tolist()

    # in case all scores are zero, I fallback to equal weights 
    if sum(weights) == 0:
        weights = None
    
    # to make the selection more random, I change the random seed per day
    anchor = top_candidates.sample(n=1, weights=weights, random_state=RANDOM_SEED + day).iloc[0]
    anchor_node = anchor['node_id']
    
    
    # STEP 4
    # as I want the user to visit several attractions per day, I also need to find some secondary attractions near
    # the daily anchor for the user to visit
    # this way, I am aiming to create somewhat coherent days, at least geographically, and a realistic route in terms of driving

    # calculate distance from anchor to all unvisited eligible attractions
    dist_from_anchor = eligible['node_id'].map(lambda n: dist_df.at[anchor_node, n])
    
    # finding secondaries within initial radius, which I have defined above to 50km in order to make the driving realistic but not 
    # too limited
    secondaries = eligible[
        (eligible.index != anchor.name) &
        (dist_from_anchor <= SECONDARY_RADIUS_KM * 1000)
    ].copy()
    
    # expand radius if not enough secondaries found, by using the expanded radius I have defined above
    if len(secondaries) < MIN_SECONDARIES:
        print(f"Only {len(secondaries)} secondaries within {SECONDARY_RADIUS_KM}km, expanding to {SECONDARY_RADIUS_EXPANDED_KM}km")
        secondaries = eligible[
            (eligible.index != anchor.name) &
            (dist_from_anchor <= SECONDARY_RADIUS_EXPANDED_KM * 1000)
        ].copy()

    # look for top secondary candidates by hiddem gem score, but to make it more interesting, I will introduce some randomness again
    top_secondary_candidates = secondaries.nlargest(TOP_N_ANCHORS, 'hiddengem_score')
    
    # by taking a weighted random sample
    if len(top_secondary_candidates) > 0:
        weights = top_secondary_candidates['hiddengem_score'].tolist()
        n_to_sample = min(MAX_STOPS_PER_DAY - 1, len(top_secondary_candidates))
        secondaries = top_secondary_candidates.sample(
            n=n_to_sample, 
            weights=weights, 
            random_state=RANDOM_SEED + day + 100,  # offset to get different results from anchor seed
            replace=False)


    # STEP 5
    # finally, with anchors and secondaries picked based on eligibility criteria, I can add them to the itinerary for each day 
    # and update where the user is

    # one limitation of my router is that I only track the driving distances between anchors, so I will estimate the daily driving
    # just to have an idea
    dist_to_anchor = distance_from_current[anchor.name]
    estimated_daily_dist = dist_to_anchor + (SECONDARY_RADIUS_KM * 1000)
    
    if estimated_daily_dist > DAILY_BUDGET_KM * 1000:
        # python
        print(f"Warning: Day {day+1} estimated driving {estimated_daily_dist/1000:.0f}km exceeds budget")
        # streamlit
        # st.warning(f"Day {day+1} estimated driving {estimated_daily_dist/1000:.0f}km exceeds budget")
        
    # first, I combine anchor and secondaries into day route
    day_route = [anchor] + secondaries.to_dict('records')
    
    # then I add all visited attraction indices to visited_ids to ensure they cannot be visited twice in the same journey
    visited_ids.add(anchor.name)
    for _, sec in secondaries.iterrows():
        visited_ids.add(sec.name)
    
    # set current node to anchor node for next day
    # (anchor is the main destination, furthest from previous position)
    current_node = anchor['node_id']

    # add anchor to tracked anchor nodes to ensure geographic diversity
    anchor_nodes_visited.append(anchor_node)
    
    # decrement remaining days
    remaining_days -= 1
    
    # add day to itinerary
    itinerary.append({
        'day': day + 1,
        'anchor': anchor['name'],
        'stops': [anchor['name']] + secondaries['name'].tolist(),
        'municipality': anchor['municipality'],
        'driving_km': distance_from_current[anchor.name] / 1000
    })

    # add distance from Keflavik to anchor to be used later when reordering the itinerary
    itinerary[-1]['anchor_dist_from_kef'] = dist_df.at[keflavik_node, anchor_node]

In [28]:
print("\n=== Your Iceland Itinerary ===\n")
for day_data in itinerary:
    print(f"Day {day_data['day']} — {day_data['municipality']}")
    for stop in day_data['stops']:
        print(f"  • {stop}")
    print(f"  Driving: ~{day_data['driving_km']:.0f}km")
    print()
print(f"Total attractions visited: {sum(len(d['stops']) for d in itinerary)}")


=== Your Iceland Itinerary ===

Day 1 — Hunabyggd
  • Reykjadalur Gönguleið
  • Kolugljufur Waterfall
  • Blonduoskirkja
  • Borgarvirki
  • Kænugarður - Kyiv Square
  Driving: ~282km

Day 2 — Akraneskaupstadur
  • Gilsárfoss
  • Snóksdalur Kirkja
  • Gústi Guðsmaður Statue
  • Vattarnesviti
  • Kópavogskirkja
  Driving: ~205km

Day 3 — Hrunamannahreppur
  • Stampar
  • Akureyri Arts Centre And Opera House
  • Hjalparfoss
  • Samkomburu Observation Deck
  • Thjorsardalur Valley
  Driving: ~130km

Day 4 — Snaefellsbaer
  • Bugsfoss
  • Statue “Sýn" by Steinunn Þórarinsdóttir
  • Lyngfellisdalur
  • Malarrif Lighthouse
  • Budahraun
  Driving: ~258km

Day 5 — Sveitarfelagid Olfus
  • Klambratun
  • Skarfabakki Harbour
  • World Sign
  • Thrihnukagigur Volcano
  • Áskirkja
  Driving: ~223km

Day 6 — Asahreppur
  • Sigoldugljufur
  • Burfell
  • Gjáin
  • Haifoss
  • Landmannalaugar
  Driving: ~119km

Day 7 — Borgarbyggd
  • Dýjafall
  • Surtshellir Lava Cave
  • Memorial of the Rural Pos

In [29]:
# the visualization below is just for debugging and development purposes and should be not be deployed

In [30]:
m = folium.Map(location=[64.9631, -19.0208], zoom_start=6)

day_colors = ['blue', 'red', 'green', 'purple', 'orange',
              'darkblue', 'darkred', 'darkgreen', 'cadetblue', 'black']

for day_data in itinerary:
    day_num = day_data['day']
    color = day_colors[(day_num - 1) % len(day_colors)]
    
    # plot anchor as star marker
    anchor_row = df_filtered[df_filtered['name'] == day_data['anchor']].iloc[0]
    folium.Marker(
        location=[anchor_row.geometry.y, anchor_row.geometry.x],
        popup=f"Day {day_num} anchor: {day_data['anchor']} ({day_data['municipality']})",
        icon=folium.Icon(color=color, icon='star', prefix='fa')
    ).add_to(m)
    
    # plot secondaries as circle markers
    for stop in day_data['stops'][1:]:  # skip anchor (first stop)
        sec_rows = df_filtered[df_filtered['name'] == stop]
        if len(sec_rows) == 0:
            continue
        sec_row = sec_rows.iloc[0]
        folium.CircleMarker(
            location=[sec_row.geometry.y, sec_row.geometry.x],
            radius=7,
            color='white',
            weight=2,
            fill_color=color,
            fill_opacity=0.8,
            popup=f"Day {day_num} secondary: {stop}"
        ).add_to(m)

# add Keflavik
folium.Marker(
    location=[63.9850, -22.6056],
    popup="Keflavik Airport",
    icon=folium.Icon(color='black', icon='plane', prefix='fa')
).add_to(m)

m

## Step 7 - Reordering the route

My routing algorithm produces a geographically somewhat diverse route including many hidden gems, but it has a big limitation, namely that it makes the user do a lot of zig zag between places. This is unnecessary, and a consequence of the many geographical constrains I tried to implement. While changing this fundamentally is outside the scope of this prototype, as it requires a change from greedy routing to a more holistic approach, I can reorder the route to try and create a more sensible order of anchors, and minimize driving a little more.
This still does not produce a perfect route, and some zig zagging is unavoidable with the technique I use, but it is a small improvement. I am using the out-and-back kind of planning method.
One limitation I am producing this way: there is no limit to how long day 1 can be, to get to the furthest anchor. Unfortunately, solving this and creating a sensible order following the first anchor is fairly complex and hard to solve without a TSP (travelling sales person) approach. Therefore, it simply needs to be acknowledged as a limitation. Let's imagine that if the person finds the day too demanding, they can pick a place along the route to stop for the night.

In [31]:
# first I sort all anchors by distance from Keflavik descending
sorted_by_dist = sorted(itinerary, key=lambda d: d['anchor_dist_from_kef'], reverse=True)

# for the outward part, I try to go to furthest first, and then work down to midpoint
# for the return it is simple, as I just try to go from midpoint back to closest
midpoint = len(sorted_by_dist) // 2
outward = sorted_by_dist[:midpoint + 1]   # furthest to middle distance
returning = sorted_by_dist[midpoint + 1:] # middle to closest

reordered = outward + returning

for i, day_data in enumerate(reordered):
    day_data['day'] = i + 1

# assign this to a new variable to be able to distinguish
reordered_itinerary = reordered

# printing this just for development
# for d in reordered_itinerary:
    # print(f"Day {d['day']}: {d['anchor']} — {d['anchor_dist_from_kef']/1000:.0f}km from KEF")

print(reordered_itinerary)

[{'day': 1, 'anchor': 'Lögmannshlíðarkirkja, Glaumbær Church', 'stops': ['Lögmannshlíðarkirkja, Glaumbær Church', 'Drangey Island', 'Grettislaug', 'Holar Cathedral', 'Reykjafoss'], 'municipality': 'Skagafjordur', 'driving_km': 263.211228, 'anchor_dist_from_kef': 350890.18400000007}, {'day': 2, 'anchor': 'Reykjadalur Gönguleið', 'stops': ['Reykjadalur Gönguleið', 'Kolugljufur Waterfall', 'Blonduoskirkja', 'Borgarvirki', 'Kænugarður - Kyiv Square'], 'municipality': 'Hunabyggd', 'driving_km': 282.2492980000001, 'anchor_dist_from_kef': 282249.2980000001}, {'day': 3, 'anchor': 'Bugsfoss', 'stops': ['Bugsfoss', 'Statue “Sýn" by Steinunn Þórarinsdóttir', 'Lyngfellisdalur', 'Malarrif Lighthouse', 'Budahraun'], 'municipality': 'Snaefellsbaer', 'driving_km': 258.46572499999996, 'anchor_dist_from_kef': 227844.70499999993}, {'day': 4, 'anchor': 'Sigoldugljufur', 'stops': ['Sigoldugljufur', 'Burfell', 'Gjáin', 'Haifoss', 'Landmannalaugar'], 'municipality': 'Asahreppur', 'driving_km': 118.6304850000

In [32]:
# for development, I can check here if distances are ok
'''
prev_node = keflavik_node
for day_data in reordered_itinerary:
    anchor_row = df_filtered[df_filtered['name'] == day_data['anchor']].iloc[0]
    anchor_node_check = anchor_row['node_id']
    leg_dist = dist_df.at[prev_node, anchor_node_check]
    print(f"Day {day_data['day']}: {day_data['anchor']} — {day_data['anchor_dist_from_kef']/1000:.0f}km from KEF, leg: {leg_dist/1000:.0f}km")
    prev_node = anchor_node_check
'''

'\nprev_node = keflavik_node\nfor day_data in reordered_itinerary:\n    anchor_row = df_filtered[df_filtered[\'name\'] == day_data[\'anchor\']].iloc[0]\n    anchor_node_check = anchor_row[\'node_id\']\n    leg_dist = dist_df.at[prev_node, anchor_node_check]\n    print(f"Day {day_data[\'day\']}: {day_data[\'anchor\']} — {day_data[\'anchor_dist_from_kef\']/1000:.0f}km from KEF, leg: {leg_dist/1000:.0f}km")\n    prev_node = anchor_node_check\n'

In [33]:
m = folium.Map(location=[64.9631, -19.0208], zoom_start=6)

day_colors = ['blue', 'red', 'green', 'purple', 'orange',
              'darkblue', 'darkred', 'darkgreen', 'cadetblue', 'black']

# draw leg lines between anchors
anchor_coords = [[63.9850, -22.6056]]  # start at Keflavik
for day_data in reordered_itinerary:
    anchor_row = df_filtered[df_filtered['name'] == day_data['anchor']].iloc[0]
    anchor_coords.append([anchor_row.geometry.y, anchor_row.geometry.x])
anchor_coords.append([63.9850, -22.6056])  # return to Keflavik

folium.PolyLine(
    locations=anchor_coords,
    color='black',
    weight=2,
    opacity=0.5,
    dash_array='5'
).add_to(m)

# draw anchors and secondaries
for day_data in itinerary:
    day_num = day_data['day']
    color = day_colors[(day_num - 1) % len(day_colors)]
    
    anchor_row = df_filtered[df_filtered['name'] == day_data['anchor']].iloc[0]
    folium.Marker(
        location=[anchor_row.geometry.y, anchor_row.geometry.x],
        popup=f"Day {day_num} anchor: {day_data['anchor']} ({day_data['municipality']})",
        icon=folium.Icon(color=color, icon='star', prefix='fa')
    ).add_to(m)
    
    for stop in day_data['stops'][1:]:
        sec_rows = df_filtered[df_filtered['name'] == stop]
        if len(sec_rows) == 0:
            continue
        sec_row = sec_rows.iloc[0]
        folium.CircleMarker(
            location=[sec_row.geometry.y, sec_row.geometry.x],
            radius=7,
            color='white',
            weight=2,
            fill_color=color,
            fill_opacity=0.8,
            popup=f"Day {day_num} secondary: {stop}"
        ).add_to(m)

folium.Marker(
    location=[63.9850, -22.6056],
    popup="Keflavik Airport",
    icon=folium.Icon(color='black', icon='plane', prefix='fa')
).add_to(m)

m

## Step 8 - Retrieve road geometries to be used for visualization

To do this, I will first extract all necessary coordinates and markers for the place I suggest to visit, and then get their road geometries so that I can make a nicer map.

In [34]:
# create needed containers
all_day_coords = []
markers = []

# loop over each day in the itinerary
for day_data in reordered_itinerary:
    day_idx = day_data['day'] - 1
    day_segments = []
    
    # build ordered list of nodes for this day
    # start from previous day's anchor node or at the airport for the first day
    if day_data['day'] == 1:
        start_node = keflavik_node
    else:
        prev_anchor = reordered_itinerary[day_idx - 1]['anchor']
        start_node = df_filtered[df_filtered['name'] == prev_anchor].iloc[0]['node_id']
    
    # get all stop nodes for this day
    day_nodes = [start_node]
    for stop in day_data['stops']:
        stop_rows = df_filtered[df_filtered['name'] == stop]
        if len(stop_rows) > 0:
            day_nodes.append(stop_rows.iloc[0]['node_id'])
    
    # get road geometry between consecutive nodes using the shortest path from networkx
    for i in range(len(day_nodes) - 1):
        origin = day_nodes[i]
        destination = day_nodes[i + 1]
        try:
            route = nx.shortest_path(G_undirected, origin, destination, weight='length')
            coords = [[G_undirected.nodes[n]['y'], G_undirected.nodes[n]['x']] for n in route]
            day_segments.extend(coords)
        except nx.NetworkXNoPath:
            print(f"No path between {origin} and {destination}")

    # collect them all
    all_day_coords.append(day_segments)
    
    # collect markers
    for stop in day_data['stops']:
        stop_rows = df_filtered[df_filtered['name'] == stop]
        if len(stop_rows) == 0:
            continue
        stop_row = stop_rows.iloc[0]
        is_anchor = stop == day_data['anchor']
        markers.append({
            'lat': stop_row.geometry.y,
            'lon': stop_row.geometry.x,
            'name': stop,
            'day': day_idx,
            'is_anchor': is_anchor
        })

# add Keflavik markers
markers.append({'lat': 63.9850, 'lon': -22.6056, 'name': 'Keflavik Airport', 'day': -1, 'is_anchor': True})

In [35]:
# just for development
print(f"Days collected: {len(all_day_coords)}")
for i, day in enumerate(all_day_coords):
    print(f"Day {i+1}: {len(day)} road coordinates")

Days collected: 10
Day 1: 334 road coordinates
Day 2: 100 road coordinates
Day 3: 74 road coordinates
Day 4: 89 road coordinates
Day 5: 48 road coordinates
Day 6: 65 road coordinates
Day 7: 149 road coordinates
Day 8: 218 road coordinates
Day 9: 309 road coordinates
Day 10: 315 road coordinates


## Step 9 - Create a folium map with the route

In [36]:
# use a map of Iceland as the base
m = folium.Map(location=[64.9631, -19.0208], zoom_start=6)

# use different colors for different days, at most we need 10 since that is the maximum number of days my prototype allows
day_colors = ['blue', 'red', 'green', 'purple', 'orange',
              'darkblue', 'darkred', 'darkgreen', 'cadetblue', 'black']

# draw road geometry per day
for day_idx, day_coords in enumerate(all_day_coords):
    color = day_colors[day_idx % len(day_colors)]
    folium.PolyLine(
        locations=day_coords,
        color=color,
        weight=3,
        opacity=0.8,
        tooltip=f"Day {day_idx + 1}"
    ).add_to(m)

# draw markers
for marker in markers:
    if marker['day'] == -1:
        folium.Marker(
            location=[marker['lat'], marker['lon']],
            popup="Keflavik Airport",
            icon=folium.Icon(color='black', icon='plane', prefix='fa')
        ).add_to(m)
    else:
        color = day_colors[marker['day'] % len(day_colors)]
        if marker['is_anchor']:
            folium.Marker(
                location=[marker['lat'], marker['lon']],
                popup=f"Day {marker['day']+1} anchor: {marker['name']}",
                icon=folium.Icon(color=color, icon='star', prefix='fa')
            ).add_to(m)
        else:
            folium.CircleMarker(
                location=[marker['lat'], marker['lon']],
                radius=7,
                color='white',
                weight=2,
                fill_color=color,
                fill_opacity=0.8,
                popup=f"Day {marker['day']+1}: {marker['name']}"
            ).add_to(m)

# show the map
folium.LayerControl().add_to(m)
m

The map looks decent. There are a few problems with nodes that were snapped to accesible roads but actually need roads that are not within my road network. I can investigate this further or simply acknowledge this limitation.

## Step 10 - Adding extras to my output

For convenience for the user, I want to also show the itinerary with some fun details.

In [37]:
# Build itinerary summary text
summary_lines = []
summary_lines.append("=== Your Iceland Alternative Route ===\n")

total_driving = 0
total_stops = 0

for day_data in reordered_itinerary:
    summary_lines.append(f"Day {day_data['day']} — {day_data['municipality']}")
    summary_lines.append(f"  Main destination: {day_data['anchor']}")
    summary_lines.append(f"  Also visiting:")
    for stop in day_data['stops'][1:]:
        summary_lines.append(f"    • {stop}")
    
    # get obscurity scores for this day
    day_scores = []
    for stop in day_data['stops']:
        stop_rows = df_filtered[df_filtered['name'] == stop]
        if len(stop_rows) > 0:
            day_scores.append(stop_rows.iloc[0]['hiddengem_score'])
    if day_scores:
        summary_lines.append(f"Average off the beaten path: {sum(day_scores)/len(day_scores) * 100:.0f}%")
    
    summary_lines.append("")
    total_driving += day_data['driving_km']
    total_stops += len(day_data['stops'])

all_scores = []
for day_data in reordered_itinerary:
    for stop in day_data['stops']:
        stop_rows = df_filtered[df_filtered['name'] == stop]
        if len(stop_rows) > 0:
            all_scores.append(stop_rows.iloc[0]['hiddengem_score'])

summary_lines.append(f"Total attractions visited: {total_stops}")
summary_lines.append(f"Overall off the beaten path: {sum(all_scores)/len(all_scores) * 100:.0f}%")

summary = "\n".join(summary_lines)
print(summary)

=== Your Iceland Alternative Route ===

Day 1 — Skagafjordur
  Main destination: Lögmannshlíðarkirkja, Glaumbær Church
  Also visiting:
    • Drangey Island
    • Grettislaug
    • Holar Cathedral
    • Reykjafoss
Average off the beaten path: 54%

Day 2 — Hunabyggd
  Main destination: Reykjadalur Gönguleið
  Also visiting:
    • Kolugljufur Waterfall
    • Blonduoskirkja
    • Borgarvirki
    • Kænugarður - Kyiv Square
Average off the beaten path: 59%

Day 3 — Snaefellsbaer
  Main destination: Bugsfoss
  Also visiting:
    • Statue “Sýn" by Steinunn Þórarinsdóttir
    • Lyngfellisdalur
    • Malarrif Lighthouse
    • Budahraun
Average off the beaten path: 61%

Day 4 — Asahreppur
  Main destination: Sigoldugljufur
  Also visiting:
    • Burfell
    • Gjáin
    • Haifoss
    • Landmannalaugar
Average off the beaten path: 48%

Day 5 — Borgarbyggd
  Main destination: Dýjafall
  Also visiting:
    • Surtshellir Lava Cave
    • Memorial of the Rural Postmen
    • Barnafoss
    • Reykholtskir